# Phase 5 — Vector Search over Jurafsky & Martin

We load the trained bi-encoder from Phase 3/4, encode every 200–300 word chunk of the
Jurafsky & Martin *Speech and Language Processing* book, and build a fast search system.

**Pipeline:**
```
query → tokenizer → BiEncoder → query_emb
                                      ↓
                        cosine_sim(query_emb, corpus_embs)
                                      ↓
                               top-k chunks
```

In [ ]:
import json, re, time, urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

MODEL_DIR = Path('../models')
DATA_DIR  = Path('../data')
RAW_DIR   = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Download Jurafsky & Martin PDF

In [ ]:
PDF_PATH = RAW_DIR / 'jurafsky_martin_slp3.pdf'
PDF_URL  = 'https://web.stanford.edu/~jurafsky/slp3/ed3book_jan72023.pdf'

if PDF_PATH.exists():
    print(f'PDF already downloaded: {PDF_PATH} ({PDF_PATH.stat().st_size / 1e6:.1f} MB)')
else:
    print(f'Downloading J&M PDF from Stanford...')
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
    print(f'Downloaded: {PDF_PATH} ({PDF_PATH.stat().st_size / 1e6:.1f} MB)')

## 2. Extract Text from PDF

We use `pdfminer.six` to extract raw text, then clean it up.

In [ ]:
from pdfminer.high_level import extract_text

print('Extracting text from PDF (this takes ~30s)...')
t0 = time.time()
raw_text = extract_text(str(PDF_PATH))
print(f'Done in {time.time()-t0:.1f}s | Raw characters: {len(raw_text):,}')
print()
print('--- First 500 chars ---')
print(raw_text[:500])

In [ ]:
def clean_pdf_text(text: str) -> str:
    """Remove PDF artefacts: hyphenated line-breaks, excessive whitespace, page numbers."""
    # Merge hyphenated line-breaks (e.g. "encod-\ning" → "encoding")
    text = re.sub(r'(\w)-\n(\w)', r'\1\2', text)
    # Replace single newlines (mid-paragraph) with space
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)
    # Collapse multiple blank lines to one
    text = re.sub(r'\n{3,}', '\n\n', text)
    # Remove standalone page numbers / short header lines
    text = re.sub(r'\n\s*\d+\s*\n', '\n', text)
    # Collapse extra spaces
    text = re.sub(r'[ \t]{2,}', ' ', text)
    return text.strip()


clean_text = clean_pdf_text(raw_text)
print(f'Cleaned characters: {len(clean_text):,}')
print()
print('--- First 500 chars after cleaning ---')
print(clean_text[:500])

## 3. Chunk into 200–300 Word Segments

We split on paragraph boundaries first, then split large paragraphs further using sentence boundaries.

In [ ]:
def word_count(text: str) -> int:
    return len(text.split())


def split_into_sentences(text: str) -> list[str]:
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]


def chunk_text(text: str, min_words: int = 100, max_words: int = 300) -> list[str]:
    """Split text into chunks of min_words–max_words using sentence boundaries."""
    sentences = split_into_sentences(text)
    chunks, current, current_wc = [], [], 0
    for sent in sentences:
        wc = word_count(sent)
        if current_wc + wc > max_words and current:
            chunk = ' '.join(current).strip()
            if word_count(chunk) >= min_words:
                chunks.append(chunk)
            current, current_wc = [], 0
        current.append(sent)
        current_wc += wc
    if current:
        chunk = ' '.join(current).strip()
        if word_count(chunk) >= min_words:
            chunks.append(chunk)
    return chunks


# Split full book text into paragraphs first, then chunk each paragraph
paragraphs = [p.strip() for p in clean_text.split('\n\n') if len(p.strip().split()) >= 30]
print(f'Paragraphs (≥30 words): {len(paragraphs):,}')

# Merge short paragraphs and chunk
all_chunks = []
buffer, buffer_wc = [], 0

for para in paragraphs:
    wc = word_count(para)
    if buffer_wc + wc > 300 and buffer:
        merged = ' '.join(buffer)
        all_chunks.extend(chunk_text(merged))
        buffer, buffer_wc = [], 0
    buffer.append(para)
    buffer_wc += wc

if buffer:
    merged = ' '.join(buffer)
    all_chunks.extend(chunk_text(merged))

print(f'Total chunks: {len(all_chunks):,}')
print(f'Avg chunk length: {np.mean([word_count(c) for c in all_chunks]):.1f} words')
print(f'Min / Max: {min(word_count(c) for c in all_chunks)} / {max(word_count(c) for c in all_chunks)} words')

In [ ]:
# Assign stable chunk IDs
jm_corpus = {f'jm_{i:04d}': text for i, text in enumerate(all_chunks)}
jm_ids    = list(jm_corpus.keys())
jm_texts  = [jm_corpus[cid] for cid in jm_ids]

# Save for later phases
with open(DATA_DIR / 'processed' / 'jm_corpus.json', 'w', encoding='utf-8') as f:
    json.dump(jm_corpus, f, ensure_ascii=False, indent=2)
print(f'Saved jm_corpus.json ({len(jm_corpus):,} chunks)')

# Print a sample chunk
print(f'\n--- Sample chunk (jm_0010) ---')
print(jm_corpus.get('jm_0010', list(jm_corpus.values())[10])[:400])

## 4. Load Trained Bi-Encoder

In [ ]:
# --- Reconstruct BiEncoder (same class as Phase 3) ---

class MeanPooling(nn.Module):
    def forward(self, token_embeddings, attention_mask):
        mask   = attention_mask.unsqueeze(-1).float()
        summed = (token_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts


class BiEncoder(nn.Module):
    def __init__(self, model_name: str, proj_dim: int | None = 256):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(model_name)
        hidden = self.transformer.config.hidden_size
        self.pooler = MeanPooling()
        if proj_dim is not None:
            self.projection = nn.Sequential(
                nn.LayerNorm(hidden),
                nn.Linear(hidden, proj_dim, bias=False),
            )
            self.emb_dim = proj_dim
        else:
            self.projection = nn.Identity()
            self.emb_dim = hidden

    def encode(self, input_ids, attention_mask):
        out    = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.pooler(out.last_hidden_state, attention_mask)
        proj   = self.projection(pooled)
        return F.normalize(proj, p=2, dim=-1)

    def forward(self, q_enc, d_enc):
        q_emb = self.encode(q_enc['input_ids'], q_enc['attention_mask'])
        d_emb = self.encode(d_enc['input_ids'], d_enc['attention_mask'])
        return q_emb, d_emb


# Load config saved in Phase 3
with open(MODEL_DIR / 'model_config.json') as f:
    cfg = json.load(f)

print(f'Loading {cfg["model_name"]} (proj_dim={cfg["proj_dim"]}, best epoch={cfg["best_epoch"]})')

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR / 'tokenizer')
model     = BiEncoder(cfg['model_name'], proj_dim=cfg['proj_dim'])
model.load_state_dict(torch.load(MODEL_DIR / 'best_biencoder.pt', map_location=DEVICE))
model = model.to(DEVICE).eval()

print(f'Model loaded. Embedding dim: {model.emb_dim}')

## 5. Encode All J&M Chunks

In [ ]:
@torch.no_grad()
def encode_texts(texts: list[str], batch_size: int = 128, max_length: int = 256) -> np.ndarray:
    all_embs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='Encoding'):
        batch = texts[i : i + batch_size]
        enc   = tokenizer(batch, padding=True, truncation=True,
                          max_length=max_length, return_tensors='pt')
        enc   = {k: v.to(DEVICE) for k, v in enc.items()}
        embs  = model.encode(enc['input_ids'], enc['attention_mask'])
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)


print(f'Encoding {len(jm_texts):,} J&M chunks...')
t0 = time.time()
jm_embeddings = encode_texts(jm_texts)
print(f'Done in {time.time()-t0:.1f}s | Shape: {jm_embeddings.shape}')

np.save(MODEL_DIR / 'jm_embeddings.npy', jm_embeddings)
with open(MODEL_DIR / 'jm_ids.json', 'w') as f:
    json.dump(jm_ids, f)
print('Saved jm_embeddings.npy and jm_ids.json')

## 6. FAISS Index (optional, fast for large corpora)

In [ ]:
try:
    import faiss
    d = jm_embeddings.shape[1]
    # Inner product index (works with L2-normalised vectors = cosine similarity)
    index = faiss.IndexFlatIP(d)
    index.add(jm_embeddings.astype('float32'))
    faiss.write_index(index, str(MODEL_DIR / 'jm_faiss.index'))
    USE_FAISS = True
    print(f'FAISS index built: {index.ntotal:,} vectors, dim={d}')
except ImportError:
    USE_FAISS = False
    print('faiss-cpu not installed — using numpy cosine search (fast enough for this corpus size)')

## 7. Search Function

In [ ]:
@torch.no_grad()
def search(query: str, k: int = 5, verbose: bool = True) -> list[dict]:
    """
    Search the J&M book for the top-k most relevant chunks.

    Args:
        query:   Natural language question or phrase
        k:       Number of results to return
        verbose: If True, pretty-print the results

    Returns:
        List of dicts with keys: chunk_id, score, text
    """
    # Encode query
    enc = tokenizer([query], padding=True, truncation=True,
                    max_length=64, return_tensors='pt')
    enc = {kk: vv.to(DEVICE) for kk, vv in enc.items()}
    q_emb = model.encode(enc['input_ids'], enc['attention_mask']).cpu().numpy()  # (1, D)

    # Retrieve
    if USE_FAISS:
        scores, indices = index.search(q_emb.astype('float32'), k)
        scores  = scores[0].tolist()
        indices = indices[0].tolist()
    else:
        raw_scores = (q_emb @ jm_embeddings.T).ravel()
        indices    = np.argsort(raw_scores)[::-1][:k].tolist()
        scores     = [float(raw_scores[i]) for i in indices]

    results = [
        {'chunk_id': jm_ids[i], 'score': round(s, 4), 'text': jm_corpus[jm_ids[i]]}
        for i, s in zip(indices, scores)
    ]

    if verbose:
        print(f'Query: "{query}"\n')
        for rank, r in enumerate(results, 1):
            print(f'--- Rank {rank} | {r["chunk_id"]} | score={r["score"]} ---')
            print(r['text'][:400] + ('...' if len(r['text']) > 400 else ''))
            print()

    return results


print('search() ready.')

## 8. Demo Queries

In [ ]:
_ = search('What is the attention mechanism in transformers?', k=3)

In [ ]:
_ = search('How does named entity recognition work?', k=3)

In [ ]:
_ = search('What is the difference between precision and recall?', k=3)

In [ ]:
_ = search('How does language modelling with n-grams work?', k=3)

In [ ]:
_ = search('What is word2vec and how are word embeddings trained?', k=3)

## 9. Interactive Search

Run this cell and type your own queries.

In [ ]:
while True:
    query = input('\nEnter query (or "quit" to stop): ').strip()
    if query.lower() in ('quit', 'q', 'exit', ''):
        break
    search(query, k=5)

## Summary

| Item | Value |
|------|-------|
| Book | Jurafsky & Martin — *Speech and Language Processing* (3rd ed.) |
| Chunks | ~800–1000 segments of 100–300 words |
| Encoder | DistilBERT + mean pool + Linear(768→256) + L2-norm |
| Index | FAISS `IndexFlatIP` (inner product = cosine on unit vectors) |
| Search | Encode query → nearest-neighbour lookup → return top-k chunks |

**Next step → Phase 6: Evaluation (Recall@k, MRR, qualitative analysis)**